# Chapter 6. Ensemble Learning and Random Forests

The fundamental philosophy of ensemble learning is the wisdom of the crowd. Aggregating the predictions of a group of diverse models often yields significantly higher accuracy and robustness than relying on the single best individual predictor.

## Voting Classifiers

A voting classifier aggregates the predictions of multiple diverse baseline models. To maximize effectiveness the baseline models should be structurally independent and utilize completely different algorithms. This ensures they make uncorrelated mathematical errors.

* **Hard Voting:** The ensemble strictly predicts the class that receives the absolute majority of votes.
* **Soft Voting:** The ensemble predicts the class with the highest average probability across all constituent models. Soft voting mathematically grants more weight to highly confident predictions and often outperforms hard voting.

**Probability Prerequisites:**
Soft voting strictly requires every single classifier in the ensemble to be capable of estimating class probabilities. Algorithms that do not output probabilities by default such as Support Vector Machines must be explicitly configured to do so before joining a soft voting ensemble.

The mathematical engine driving this is the Law of Large Numbers. Even if individual models are weak learners barely exceeding 51% accuracy an ensemble of 1000 independent models can mathematically push the aggregated accuracy closer to 75%.

First let's construct a Hard Voting ensemble to establish a performance baseline. We will evaluate the individual models and observe how the majority vote mechanism operates.

In [1]:
from sklearn.datasets import make_moons
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

# Generate and split a nonlinear toy dataset
X, y = make_moons(n_samples=500, noise=0.30, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# Train Hard Voting Classifier
voting_clf = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(random_state=42)),
        ('rf', RandomForestClassifier(random_state=42)),
        ('svc', SVC(random_state=42))
    ]
)
voting_clf.fit(X_train, y_train)

# Evaluate individual baseline models
print('Individual accuracy')
for name, clf in voting_clf.named_estimators_.items():
    print(name, '=', clf.score(X_test, y_test))

Individual accuracy
lr = 0.864
rf = 0.896
svc = 0.896


In [23]:
# Show hard voting mechanism for the first instance
print('Ensemble Prediction \n', voting_clf.predict(X_test[:1]))
print('Individual Predictions \n', [clf.predict(X_test[:1]) for clf in voting_clf.estimators_])

Ensemble Prediction 
 [1]
Individual Predictions 
 [array([1], dtype=int64), array([1], dtype=int64), array([0], dtype=int64)]


In [25]:
# Evaluate Hard Voting Ensemble
print('Hard Voting Accuracy \n', voting_clf.score(X_test, y_test))

Hard Voting Accuracy 
 0.912


The voting classifier predicts class 1 for the first instance of the test set, because two out of three classifiers predict that class.

Now let's try Soft Voting to see if using probability estimates improves our accuracy. Since Soft Voting relies on confidence levels rather than just a strict majority count we first need to explicitly enable probability calculations for our SVM classifier.

In [35]:
# Upgrade to Soft Voting dynamically
voting_clf.voting = 'soft'
voting_clf.named_estimators['svc'].probability = True
voting_clf.fit(X_train, y_train)

# Evaluate Soft Voting Ensemble
print('Soft Voting Accuracy \n', voting_clf.score(X_test, y_test))

Soft Voting Accuracy 
 0.92


## Bagging and Pasting

Instead of utilizing different algorithms we can use the exact same training algorithm but train each independent predictor on a randomly sampled subset of the training data. This architecture allows models to be trained entirely in parallel across multiple CPU cores making it highly scalable.

* **Bagging:** Sampling training instances with replacement. This introduces more diversity resulting in a slightly higher bias but significantly lower variance. Bagging generally yields superior generalized models.
* **Pasting:** Sampling training instances strictly without replacement.

**Bias-Variance Tradeoff**  
Each individual predictor has a higher bias than if it were trained on the entire original dataset. However aggregating them mathematically reduces both bias and variance. Bagging is typically preferred in production because the extra diversity reduces the ensemble's variance even further.

In practice, the ensemble often ends up with a similar bias but a lower variance than a single predictor trained on the original training set. Therefore it works best with high-variance and low-bias models (e.g., decision trees).

### Bagging and Pasting in Scikit-Learn

Scikit-Learn provides a simple API for both bagging and pasting through:

- `BaggingClassifier` (for classification)
- `BaggingRegressor` (for regression)

The following example trains an ensemble of 500 Decision Tree classifiers, each trained on 100 training instances.

In [26]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

bag_clf = BaggingClassifier(DecisionTreeClassifier(), n_estimators=500,
                            max_samples=100, n_jobs=-1, random_state=42)
bag_clf.fit(X_train, y_train)

print('Bagging Ensemble Accuracy \n', bag_clf.score(X_test, y_test))

Bagging Ensemble Accuracy 
 0.904


This is an example of bagging, but if you want to use pasting instead, just set `bootstrap=False`.

**Note:** `BaggingClassifier` automatically performs soft voting when the base classifier supports `predict_proba()`.

### Out of Bag Evaluation

When utilizing Bagging with replacement mathematical probability dictates that only about 63% of the training instances are uniquely sampled for any given predictor. The untouched 37% of instances are structurally isolated and called Out of Bag instances.

Because these instances were never seen by the predictor during training they can be evaluated to calculate a highly accurate and unbiased validation score without actually sacrificing any valuable data to create a separate validation set.

In Scikit-Learn, you can set `oob_score=True` when creating a `BaggingClassifier` to request an automatic OOB evaluation after training. The resulting evaluation score is available in the `oob_score_` attribute.

In [42]:
bag_clf_oob = BaggingClassifier(DecisionTreeClassifier(), n_estimators=500,
                            oob_score=True, n_jobs=-1, random_state=42)
bag_clf_oob.fit(X_train, y_train)
print('Out of Bag Evaluation Score \n', bag_clf_oob.oob_score_)

Out of Bag Evaluation Score 
 0.896


In [44]:
from sklearn.metrics import accuracy_score
y_pred = bag_clf_oob.predict(X_test)
print('Accuracy on the test set \n', accuracy_score(y_test, y_pred))

Accuracy on the test set 
 0.92


### Random Patches and Random Subspaces

The BaggingClassifier natively supports sampling features alongside instances. This is exceptionally useful for high dimensional inputs like images. 
* **Random Patches method**: Sampling both instances and features  
* **Random Subspaces method**: Keeping all training instances but sampling features

Sampling features results in even more predictor diversity, trading a bit more bias for a lower variance.